# F02_01 — Exploración de series temporales

## Propósito de la sesión

Esta sesión inicia formalmente el bloque de **Forecasting**. El objetivo todavía no es ajustar un modelo de pronóstico, sino aprender a reconocer la estructura temporal que un futuro modelo debería ser capaz de representar.

Al finalizar la sesión, el estudiante deberá poder:

1. identificar la variable temporal, la frecuencia y la variable objetivo;
2. construir e interpretar un gráfico temporal;
3. distinguir tendencia, estacionalidad y ciclos;
4. construir e interpretar rezagos;
5. calcular e interpretar la función de autocorrelación (ACF);
6. reconocer la idea de ruido blanco;
7. conectar la dependencia temporal de una serie con el problema de pronóstico.

> **Idea central:** antes de pronosticar, debemos comprender qué información del pasado parece persistir en el futuro.

### Cómo trabajar este notebook

En varias secciones se repite la siguiente secuencia:

- 📘 **Concepto**
- 🌍 **Contexto de la serie**
- 💻 **Entendamos el código**
- 🧠 **Predice**
- ▶ **Ejecuta**
- 🔎 **Interpreta**
- ▶ **Ver respuesta razonada**
- 🛠️ **Modifica**


## 1. Preparación del notebook

Cada notebook del curso es **autocontenido**: importa explícitamente las librerías que necesita.

`F00_configuracion_forecasting.ipynb` sirve para comprobar que el entorno está correctamente instalado y que los datos del curso están disponibles, pero sus variables e imports no se comparten automáticamente con otros notebooks.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.tsa.stattools import acf

DATA_DIR = Path("data/fpppy")

assert DATA_DIR.exists(), (
    "No se encuentra la carpeta data/fpppy. "
    "Verifique la ubicación del notebook y la carpeta de datos."
)

sns.set_style("whitegrid")


<details>
<summary><strong>💻 ¿Qué hace este bloque?</strong></summary>

- `Path` permite trabajar con rutas de archivos de forma legible.
- `numpy` se utiliza para operaciones numéricas.
- `pandas` permite cargar y transformar datos tabulares.
- `matplotlib` y `seaborn` permiten construir gráficos.
- `plot_acf` permite representar la función de autocorrelación.
- `DATA_DIR = Path("data/fpppy")` define una única ruta base para los datos.
- El `assert` verifica desde el inicio que la carpeta exista.

</details>


## 2. El gráfico temporal como punto de partida

### 📘 Concepto

Una serie temporal no es simplemente una columna de números: **el orden de las observaciones importa**.

Un gráfico temporal permite detectar visualmente:

- cambios de nivel;
- crecimiento o disminución persistente;
- patrones repetitivos;
- observaciones inusuales;
- cambios en la variabilidad.


### 🌍 Contexto de la serie — Ansett

El conjunto `ansett.csv` contiene observaciones de tráfico aéreo de **Ansett Australia**.

En este notebook trabajaremos con una sola serie:

- **Ruta:** MEL–SYD.
- **Clase:** Economy.
- **Frecuencia:** semanal.
- **Variable analizada:** número de pasajeros.
- **Uso didáctico:** introducir gráficos temporales, rezagos y autocorrelación.

> No todas las series se utilizan para ilustrar lo mismo. Elegimos esta serie porque su estructura resulta útil para estudiar dependencia temporal de corto plazo.


### 💻 Entendamos el código

```python
ansett = pd.read_csv(DATA_DIR / "ansett.csv", parse_dates=["ds"])
```

- `pd.read_csv(...)` lee un archivo CSV.
- `DATA_DIR / "ansett.csv"` construye la ruta `data/fpppy/ansett.csv`.
- `parse_dates=["ds"]` indica que `ds` debe tratarse como fecha.

Después seleccionaremos una sola ruta aérea y una sola clase.


### 🧠 Predice

1. ¿Qué tipo de objeto esperas obtener de `pd.read_csv(...)`?
2. Si filtramos por una ruta y una clase, ¿esperas obtener más o menos filas que en el archivo original?
3. ¿Por qué será importante ordenar las observaciones por fecha antes de graficar?

<details>
<summary><strong>▶ Ver respuesta razonada</strong></summary>

1. `pd.read_csv(...)` devuelve un `DataFrame`.
2. El filtrado debe producir menos filas.
3. En una serie temporal, el orden cronológico es parte de la información.

</details>


In [ ]:
ansett = pd.read_csv(DATA_DIR / "ansett.csv", parse_dates=["ds"])

serie = (
    ansett
    .loc[(ansett["Airports"] == "MEL-SYD") & (ansett["Class"] == "Economy")]
    .sort_values("ds")
    .copy()
)

serie.head()


### 💻 Lea el pipeline

```python
ansett.loc[...]
```
selecciona solo las filas de interés.

```python
.sort_values("ds")
```
ordena cronológicamente.

```python
.copy()
```
crea una copia independiente del subconjunto seleccionado.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

ax.plot(serie["ds"], serie["y"])

ax.set(
    title="Pasajeros semanales — MEL-SYD, clase Economy",
    xlabel="Semana",
    ylabel="Pasajeros"
)

plt.show()


### 🔎 Interpreta

1. ¿La serie presenta un nivel aproximadamente constante?
2. ¿Observa tendencia?
3. ¿Hay cambios estructurales o valores inusuales?
4. ¿Parece existir un patrón que se repite con periodicidad fija?

<details>
<summary><strong>▶ Ver respuesta razonada</strong></summary>

La respuesta debe justificarse con evidencia visual.

- Una **tendencia** refleja un movimiento persistente de largo plazo.
- Una posible **estacionalidad** requiere repetición con periodicidad fija.
- Cambios abruptos no deben confundirse automáticamente con tendencia o estacionalidad.

</details>


### 🛠️ Modifica

Cambie temporalmente `"Economy"` por otra categoría disponible en `Class`.

Compare la nueva serie con la original:

- ¿cambia el nivel?
- ¿cambia la variabilidad?
- ¿se conserva la misma estructura temporal?

Después restaure `"Economy"`.


## 3. Tendencia, estacionalidad y ciclos

### 📘 Concepto

- **Tendencia:** movimiento de largo plazo ascendente o descendente.
- **Estacionalidad:** patrón que se repite con un período fijo y conocido.
- **Ciclo:** fluctuación de duración no fija; no está necesariamente ligada al calendario.


### 🌍 Contexto de la serie — PBS

El archivo `PBS_unparsed.csv` contiene información mensual asociada al **Pharmaceutical Benefits Scheme (PBS) de Australia**.

En este ejercicio trabajaremos con:

- **Grupo:** `ATC2 = "A10"`.
- **Frecuencia:** mensual.
- **Variable analizada:** costo agregado mensual.
- **Uso didáctico:** observar una serie donde tendencia y estacionalidad son visualmente relevantes.

Esta serie contrasta con la anterior: aquí buscamos reconocer componentes de largo plazo y patrones que se repiten regularmente.


### 💻 Entendamos el código

```python
pbs.loc[pbs["ATC2"] == "A10"]
```
filtra únicamente el grupo `A10`.

```python
.groupby("Month", as_index=False)["Cost"].sum()
```
agrupa por mes y suma el costo.

```python
.sort_values("Month")
```
recupera el orden temporal.


### 🧠 Predice

¿Qué problema podría aparecer si agrupamos correctamente por mes pero después graficamos los meses en un orden no cronológico?

<details>
<summary><strong>▶ Ver respuesta razonada</strong></summary>

El gráfico conectaría puntos en un orden incorrecto. Esto puede fabricar cambios artificiales y destruir patrones reales.

</details>


In [ ]:
pbs = pd.read_csv(DATA_DIR / "PBS_unparsed.csv", parse_dates=["Month"])

a10 = (
    pbs.loc[pbs["ATC2"] == "A10"]
    .groupby("Month", as_index=False)["Cost"]
    .sum()
    .sort_values("Month")
)

a10.head()


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

ax.plot(a10["Month"], a10["Cost"])

ax.set(
    title="Costo mensual agregado — grupo A10",
    xlabel="Mes",
    ylabel="Costo"
)

plt.show()


### 🔎 Interpreta

¿Qué componentes parecen estar presentes en esta serie?

No responda únicamente “hay tendencia” o “hay estacionalidad”. Describa **qué observa** que respalda su afirmación.

<details>
<summary><strong>▶ Ver respuesta razonada</strong></summary>

La serie muestra un crecimiento de largo plazo y oscilaciones que se repiten aproximadamente con periodicidad anual. Existen indicios visuales tanto de tendencia como de estacionalidad.

</details>


### 🛠️ Modifica

Cambie el grupo `A10` por otro valor disponible en `ATC2`.

Compare visualmente ambas series:

- ¿mantienen el mismo nivel?
- ¿mantienen el mismo patrón temporal?
- ¿sería razonable asumir que una misma estructura de modelo funciona igual para ambas?


## 4. Rezagos

### 📘 Concepto

Para una serie $y_t$, el **rezago 1** compara $y_t$ con $y_{t-1}$; el rezago 2 compara $y_t$ con $y_{t-2}$, etc.

Si existe una relación visible entre $y_t$ y sus valores rezagados, el pasado contiene información sobre el presente.


### 💻 Entendamos `shift()`

```python
serie["y"].shift(1)
```

`shift(1)` desplaza la serie una posición. Así podemos colocar en una misma fila $y_t$ y $y_{t-1}$.

Con:

```python
serie["y"].shift(2)
```

obtenemos $y_{t-2}$.


### 🧠 Predice

Si una serie tiene 100 observaciones:

1. ¿cuántos valores faltantes produce `shift(1)`?
2. ¿cuántos produce `shift(3)`?
3. ¿por qué?

<details>
<summary><strong>▶ Ver respuesta razonada</strong></summary>

- `shift(1)` produce 1 valor faltante.
- `shift(3)` produce 3 valores faltantes.

Las primeras observaciones no tienen suficientes valores anteriores disponibles para construir esos rezagos.

</details>


In [ ]:
lag_df = pd.DataFrame({
    "y_t": serie["y"],
    "y_t_minus_1": serie["y"].shift(1),
    "y_t_minus_2": serie["y"].shift(2),
}).dropna()

lag_df.head()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].scatter(lag_df["y_t_minus_1"], lag_df["y_t"], alpha=0.6)
axes[0].set(title="Lag 1", xlabel="$y_{t-1}$", ylabel="$y_t$")

axes[1].scatter(lag_df["y_t_minus_2"], lag_df["y_t"], alpha=0.6)
axes[1].set(title="Lag 2", xlabel="$y_{t-2}$", ylabel="$y_t$")

plt.show()


### 🔎 Interpreta

1. ¿Cuál de los dos gráficos parece mostrar una relación más fuerte?
2. ¿Qué significaría una nube de puntos sin estructura visible?
3. ¿Una relación fuerte en un lag plot implica causalidad?

<details>
<summary><strong>▶ Ver respuesta razonada</strong></summary>

- Una estructura lineal más definida sugiere mayor dependencia lineal en ese rezago.
- Una nube sin estructura sugiere poca relación lineal.
- No implica causalidad; describe asociación temporal.

</details>


### 🛠️ Modifica

Agregue un rezago 3 con:

```python
serie["y"].shift(3)
```

y construya un nuevo lag plot. Compare visualmente los rezagos 1, 2 y 3.


## 5. Autocorrelación y ACF

### 📘 Concepto

La **autocorrelación** cuantifica la relación lineal entre una serie y versiones rezagadas de sí misma.

La **función de autocorrelación (ACF)** muestra simultáneamente la autocorrelación para varios rezagos.

El rezago 0 no resulta informativo aquí porque una serie correlacionada consigo misma tiene correlación 1.


### 💻 Entendamos `plot_acf`

```python
plot_acf(
    serie["y"],
    lags=30,
    zero=False
)
```

- `serie["y"]`: variable analizada.
- `lags=30`: muestra hasta 30 rezagos.
- `zero=False`: omite el rezago 0.


### 🧠 Predice

A partir de los lag plots:

- ¿esperas que la autocorrelación del rezago 1 sea positiva, negativa o cercana a cero?
- ¿esperas que la dependencia desaparezca inmediatamente o gradualmente?

<details>
<summary><strong>▶ Ver respuesta razonada</strong></summary>

Si el lag plot 1 muestra una relación positiva clara, esperamos una autocorrelación positiva en el primer rezago. Si los rezagos siguientes conservan estructura, la dependencia debería disminuir gradualmente.

</details>


In [ ]:
acf_values = acf(serie["y"].dropna(), nlags=10)

for lag, value in enumerate(acf_values[:6]):
    print(f"Lag {lag}: {value:.3f}")


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))

plot_acf(
    serie["y"].dropna(),
    lags=30,
    zero=False,
    bartlett_confint=False,
    auto_ylims=True,
    ax=ax
)

ax.set(
    title="ACF — pasajeros semanales MEL-SYD",
    xlabel="Rezago",
    ylabel="Autocorrelación"
)

plt.show()


### 🔎 Interpreta

1. ¿Qué signo tiene la autocorrelación de primer orden?
2. ¿La dependencia desaparece rápidamente o persiste?
3. ¿Se observan picos periódicos?
4. ¿Qué implicaría esta ACF para un futuro modelo de pronóstico?

<details>
<summary><strong>▶ Ver respuesta razonada</strong></summary>

Valores importantes en los primeros rezagos indican que observaciones cercanas en el tiempo contienen información entre sí. Si las autocorrelaciones disminuyen gradualmente, existe persistencia temporal.

</details>


### 🛠️ Modifica

Cambie `lags=30` por `lags=15` y luego por `lags=50`.

Observe cómo cambia la cantidad de rezagos mostrados, pero no los valores ya calculados para los rezagos comunes.


## 6. Ruido blanco

### 📘 Concepto

En este curso utilizaremos como referencia una serie de **ruido blanco** con media y varianza aproximadamente constantes y sin autocorrelaciones sistemáticas.

En Forecasting esta idea es importante porque:

1. una serie de ruido blanco contiene poca estructura temporal lineal aprovechable mediante sus propios rezagos;
2. más adelante, después de ajustar un modelo, queremos que sus **residuos** se parezcan razonablemente a ruido blanco.


### 🌍 Contexto de la serie — ejemplo simulado

Esta serie **no proviene de un sistema real**. La generaremos artificialmente.

- **Variable:** valores aleatorios con media 0.
- **Frecuencia:** índice temporal artificial.
- **Uso didáctico:** mostrar cómo luce una serie sin estructura temporal sistemática.


### 🧠 Predice

Si generamos números aleatorios independientes:

1. ¿esperas una tendencia clara?
2. ¿esperas autocorrelaciones grandes en muchos rezagos?
3. ¿qué aspecto debería tener su ACF?

<details>
<summary><strong>▶ Ver respuesta razonada</strong></summary>

No esperamos tendencia sistemática ni autocorrelaciones persistentes. La mayoría deberían estar alrededor de cero, aunque por azar algunas pueden alejarse ligeramente.

</details>


In [ ]:
rng = np.random.default_rng(30)
white_noise = rng.normal(0, 1, 120)

fig, axes = plt.subplots(2, 1, figsize=(9, 6))

axes[0].plot(np.arange(1, len(white_noise) + 1), white_noise)
axes[0].set(title="Ejemplo simulado de ruido blanco", xlabel="Tiempo", ylabel="Valor")

plot_acf(
    white_noise,
    lags=25,
    zero=False,
    bartlett_confint=False,
    auto_ylims=True,
    ax=axes[1]
)

axes[1].set(title="ACF del ruido blanco", xlabel="Rezago", ylabel="Autocorrelación")

plt.tight_layout()
plt.show()


### 🔎 Interpreta

Compare esta ACF con la ACF de la serie de pasajeros.

<details>
<summary><strong>▶ Ver respuesta razonada</strong></summary>

La serie simulada no muestra un patrón persistente de autocorrelaciones. En cambio, una serie con dependencia temporal suele mostrar valores sistemáticamente distintos de cero en algunos rezagos.

</details>


## 7. Conexión con los proyectos integradores

### 📘 Dos descripciones complementarias del sistema

Cada proyecto contiene:

- $X_t$: un **estado** del sistema, modelado mediante una cadena de Markov;
- $Y_t$: una **variable cuantitativa observada semanalmente**, analizada como serie temporal.

En esta semana **no se pronostica todavía**. El objetivo es caracterizar $Y_t$.


### 🌍 Variables de los cuatro casos

| Caso | Estado $X_t$ | Variable temporal $Y_t$ |
|---|---|---|
| Mantenimiento | estado del motor | `vibracion_rms` |
| Calidad | estado del proceso | `porcentaje_no_conformes` |
| Servicios | estado operativo | `solicitudes` |
| Plataforma | estado de engagement | `usuarios_activos` |

> Aunque $X_t$ y $Y_t$ estén relacionados, describen aspectos diferentes del sistema.


### 🧠 Pregunta de integración

Suponga que dos semanas consecutivas presentan el mismo estado $X_t$, pero valores distintos de $Y_t$.

¿Eso contradice el modelo del proyecto?

<details>
<summary><strong>▶ Ver respuesta razonada</strong></summary>

No. El estado representa una categoría operativa, mientras que $Y_t$ es una cantidad numérica asociada al funcionamiento del sistema. Un mismo estado puede corresponder a diferentes valores de la variable cuantitativa.

</details>


### 7.1 Comparación de una variable cuantitativa entre estados

### 📘 Concepto

En los proyectos integradores, $X_t$ representa un **estado del sistema** y $Y_t$ una **variable cuantitativa asociada a su funcionamiento**.

Una forma inicial de estudiar su relación consiste en comparar descriptivamente la distribución de $Y_t$ entre los distintos estados de $X_t$.

Esto permite responder preguntas como:

- ¿el nivel medio de $Y_t$ cambia entre estados?;
- ¿la variabilidad es similar?;
- ¿hay solapamiento entre las distribuciones?;
- ¿un mismo estado puede presentar distintos valores de $Y_t$?

> Esta comparación es **descriptiva**. Asociación no implica causalidad y tampoco significa que $X_t$ determine completamente a $Y_t$.


### 🌍 Contexto del ejemplo — sistema sintético

Construiremos un ejemplo didáctico independiente de los cuatro proyectos.

- **Estado 0:** operación normal.
- **Estado 1:** alerta.
- **Estado 2:** condición degradada.
- **Variable cuantitativa:** indicador semanal de desempeño.
- **Uso didáctico:** practicar `groupby()` y una comparación gráfica antes de aplicar estas herramientas al proyecto integrador.


In [ ]:
rng = np.random.default_rng(42)

estado = np.repeat([0, 1, 2], 40)
indicador = np.concatenate([
    rng.normal(50, 4, 40),
    rng.normal(60, 5, 40),
    rng.normal(72, 6, 40),
])

ejemplo_estados = pd.DataFrame({
    "estado": estado,
    "indicador": indicador
})

ejemplo_estados.head()


### 💻 Entendamos el código

La expresión:

```python
ejemplo_estados.groupby("estado")["indicador"]
```

agrupa las observaciones según el valor de `estado`.

Luego:

```python
.agg(["count", "mean", "std"])
```

calcula, para cada estado:

- número de observaciones;
- media;
- desviación estándar.

Esto produce un resumen descriptivo comparable entre grupos.


### 🧠 Predice

Antes de ejecutar el resumen:

1. ¿esperas que los tres estados tengan la misma media?
2. ¿esperas que un mismo estado tenga un único valor posible del indicador?
3. si las medias son diferentes, ¿eso demuestra que el estado causa el valor del indicador?

<details>
<summary><strong>▶ Ver respuesta razonada</strong></summary>

1. No. El ejemplo fue construido con niveles medios diferentes entre estados.
2. No. Cada estado contiene variabilidad interna, por lo que aparecen distintos valores del indicador.
3. No. Una diferencia descriptiva entre grupos muestra asociación, no causalidad.

</details>


In [ ]:
resumen_estados = (
    ejemplo_estados
    .groupby("estado")["indicador"]
    .agg(["count", "mean", "std"])
)

resumen_estados


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

sns.boxplot(
    data=ejemplo_estados,
    x="estado",
    y="indicador",
    ax=ax
)

ax.set(
    title="Distribución del indicador por estado",
    xlabel="Estado",
    ylabel="Indicador semanal"
)

plt.show()


### 🔎 Interpreta

1. ¿cómo cambia el nivel central del indicador entre estados?
2. ¿existe variabilidad dentro de cada estado?
3. ¿hay solapamiento entre las distribuciones?
4. ¿por qué sería incorrecto afirmar que `estado` e `indicador` son la misma variable?

<details>
<summary><strong>▶ Ver respuesta razonada</strong></summary>

Las distribuciones presentan niveles centrales diferentes, pero también variabilidad dentro de cada estado. Esto muestra que un mismo estado puede asociarse con distintos valores de la variable cuantitativa.

`estado` es una categoría del sistema; `indicador` es una medición numérica. Están relacionados en este ejemplo, pero no son equivalentes.

</details>


### 🛠️ Modifica

Cambie temporalmente la media o la desviación estándar de uno de los estados en la celda donde se genera `indicador`.

Vuelva a ejecutar el resumen y el boxplot.

Observe cómo cambian:

- la separación entre estados;
- la variabilidad;
- el solapamiento entre distribuciones.


## 8. Cierre

Antes de elegir un modelo de forecasting debemos poder responder:

> **¿Qué estructura temporal existe en los datos que el modelo debería ser capaz de representar?**

En la siguiente semana utilizaremos esta caracterización para construir pronósticos de referencia y aprender a evaluarlos fuera de muestra.

### Verificación personal

Al terminar, debería poder explicar con sus propias palabras:

- qué es un gráfico temporal;
- cómo distinguir tendencia, estacionalidad y ciclo;
- qué representa un rezago;
- qué mide una autocorrelación;
- cómo interpretar una ACF;
- qué significa ruido blanco;
- por qué estas ideas son necesarias antes de ajustar modelos.
